# 05 Attention Lab：Q / K / V / Mask

本 Notebook 实现 scaled dot-product attention：

```text
scores = QK^T / sqrt(d_k)
weights = softmax(scores)
output = weights V
```

第五周先掌握机制；第六周再扩展到 Self-Attention 与 Multi-Head Attention。


In [ ]:
import math
import torch

torch.manual_seed(42)

batch_size = 2
num_queries = 3
num_keys = 4
d_k = 5
d_v = 6

Q = torch.randn(batch_size, num_queries, d_k)
K = torch.randn(batch_size, num_keys, d_k)
V = torch.randn(batch_size, num_keys, d_v)

scores = Q @ K.transpose(-2, -1)
scores = scores / math.sqrt(d_k)

weights = torch.softmax(scores, dim=-1)
output = weights @ V

print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)
print("scores:", scores.shape)
print("weights:", weights.shape)
print("output:", output.shape)

## Attention weights 在 key 维度上和为 1

In [ ]:
weights.sum(dim=-1)

## 加入 Padding Mask

In [ ]:
# 第一个样本只有前 2 个 key 有效；
# 第二个样本 4 个 key 都有效。
valid_mask = torch.tensor([
    [True, True, False, False],
    [True, True, True, True],
])

# [B,K] -> [B,1,K]，广播到每个 query。
expanded_mask = valid_mask.unsqueeze(1)

masked_scores = scores.masked_fill(
    ~expanded_mask,
    -1e9,
)

masked_weights = torch.softmax(
    masked_scores,
    dim=-1,
)

masked_output = masked_weights @ V

print(masked_weights)
print("masked output:", masked_output.shape)

## 检查第一个样本 padding 权重

In [ ]:
print(masked_weights[0, :, 2:])
print("应当接近 0")

## 关键理解

- `QK^T`：Query 与每个 Key 的匹配分数；
- Softmax：把分数变成概率式权重；
- `weights @ V`：根据权重汇总真正的信息；
- Mask：让 `<pad>` 等无效位置无法被关注。

下一周的 Self-Attention 只是进一步让 Q/K/V 都来自同一条序列。
